# Notebook 2: Model Architectures

**What:** Demucs, HDemucs, and HTDemucs — structure, parameters, input/output shapes.

**Why:** Choosing and modifying architectures requires understanding their building blocks (U-Net encoder-decoder, hybrid branches, Transformer).

**How:** Instantiate models with small configs, run forward passes, compare sizes.

## 1. Common Setup

All models expect: `(batch, channels, samples)` and return `(batch, sources, channels, samples)`.

In [1]:
from os import path
import sys
sys.path.insert(0, r'D:\demucs')

import torch
from demucs.demucs import Demucs
from demucs.hdemucs import HDemucs
from demucs.htdemucs import HTDemucs
import torchaudio
from pathlib import Path

sources = ['drums', 'bass', 'other', 'vocals']
sr = 44100
segment_sec = 4  # Short for memory (3070 Ti)
batch = 2
x = torch.randn(batch, 2, sr * segment_sec)  # (B, C, T)

print(f"Input shape: {x.shape}")

Input shape: torch.Size([2, 2, 176400])


## 2. Demucs (Waveform U-Net)

**What:** Encoder-decoder over waveform. Strided conv downsamples, transposed conv upsamples. Optional LSTM, DConv residual branches.

**Why waveform:** No STFT phase issues; end-to-end learning.

Use smaller `channels` and `depth` for 3070 Ti.

In [2]:
# ─── WHAT: Build a Demucs waveform U-Net and run a forward pass ─────────────────
# Demucs: encoder-decoder over raw waveform. Strided conv downsamples time,
# transposed conv upsamples. Outputs one waveform per source (drums, bass, etc.).
# WHY: We need a model that maps (B, C, T) mixed audio → (B, S, C, T) separated stems.
#      Waveform-level avoids STFT phase issues; end-to-end learning.
# HOW: Instantiate with config, count params, run dummy forward to verify shapes.

model_demucs = Demucs(
    sources=sources,           # WHAT: Output names ['drums','bass','other','vocals'] → 4 stems.
    #                          WHY: Defines output dim; model learns to assign each sample to a source.
    audio_channels=2,         # WHAT: Input/output stereo (L+R). WHY: Match typical song format.
    channels=32,               # WHAT: Base channel count in first conv (default 64).
    #                          WHY: Controls capacity; 32 reduces VRAM for 3070 Ti.
    depth=6,                  # WHAT: Number of encoder/decoder layers (default 6).
    #                          WHY: Each layer halves time dim; depth=4 → ~16× downsampling.
    kernel_size=8,            # WHAT: Conv kernel length in samples.
    #                          WHY: Affects receptive field; must align with stride.
    stride=4,                 # WHAT: Down/upsample factor per layer (time ÷4, then ×4).
    #                          WHY: kernel_size/stride=2 → ~50% overlap; balance speed vs resolution.
    segment=segment_sec,      # WHAT: Preferred chunk length in seconds for inference.
    #                          WHY: apply_model uses this to chunk long audio; training samples this length.
    dconv_attn=3, # depth is small, activate the deepest layer with attn
    dconv_lstm=3, # depth is small, activate the deepest layer with lstm
    dconv_mode=3
)

# WHAT: Count trainable parameters. WHY: Compare model size, estimate VRAM.
n_params = sum(p.numel() for p in model_demucs.parameters())
print(f"Demucs params: {n_params / 1e6:.2f}M")

# WHAT: Forward pass without gradients (inference mode).
# WHY: Test that input/output shapes match; avoid OOM during architecture exploration.
with torch.no_grad():
    out = model_demucs(x)
print(f"Output shape: {out.shape}  # (batch, sources, channels, samples)")

Demucs params: 45.36M
Output shape: torch.Size([2, 4, 2, 176400])  # (batch, sources, channels, samples)


## 3. HDemucs (Hybrid)

**What:** Two parallel branches — time (waveform) and frequency (spectrogram). They merge at bottleneck, then split again in decoder.

**Why hybrid:** Frequency branch captures harmonics; time branch preserves phase.

In [3]:
model_hdemucs = HDemucs(
    sources=sources,
    audio_channels=2,
    channels=32,
    depth=6,
    segment=segment_sec,
)

n_params = sum(p.numel() for p in model_hdemucs.parameters())
print(f"HDemucs params: {n_params / 1e6:.2f}M")

with torch.no_grad():
    out = model_hdemucs(x)
print(f"Output shape: {out.shape}")

HDemucs params: 37.22M
Output shape: torch.Size([2, 4, 2, 176400])


## 4. HTDemucs (Hybrid + Transformer)

**What:** Same hybrid structure, but encoder outputs feed a CrossTransformer before decoder. Self-attention + cross-attention across time/freq.

**Why Transformer:** Better long-range modeling than pure convolution.

**Note:** HTDemucs has `segment` built-in; max ~7.8s for default config.

In [ ]:
model_htdemucs = HTDemucs(
    sources=sources,
    audio_channels=2,
    channels=32,
    depth=4,
    t_layers=3,       # default 5; reduce for memory
    t_heads=4,
    segment=segment_sec,
)

n_params = sum(p.numel() for p in model_htdemucs.parameters())
print(f"HTDemucs params: {n_params / 1e6:.2f}M")

with torch.no_grad():
    out = model_htdemucs(x)
print(f"Output shape: {out.shape}")

HTDemucs params: 45.41M
Output shape: torch.Size([2, 4, 2, 176400])


## 5. Parameter Comparison

Small configs for 3070 Ti. Full HTDemucs (channels=48, t_layers=5) is larger.

In [12]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

print("Model          | Params (M)")
print("---------------+------------")
print(f"Demucs (32ch)  | {count_params(model_demucs)/1e6:.2f}")
print(f"HDemucs (32ch) | {count_params(model_hdemucs)/1e6:.2f}")
print(f"HTDemucs (32ch)| {count_params(model_htdemucs)/1e6:.2f}")

Model          | Params (M)
---------------+------------
Demucs (32ch)  | 45.36
HDemucs (32ch) | 37.22
HTDemucs (32ch)| 45.41


## 6. Memory Check on GPU

Run one forward pass and check VRAM usage.

In [13]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    x_gpu = x.to(device)
    model_htdemucs.to(device)
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = model_htdemucs(x_gpu)
    mb = torch.cuda.max_memory_allocated() / 1e6
    print(f"HTDemucs forward pass: ~{mb:.0f} MB peak VRAM")
else:
    print("CUDA not available; run on CPU only")

HTDemucs forward pass: ~565 MB peak VRAM


**Next:** Notebook 3 — Inference pipeline (apply_model, chunking, separation).